# Patient-Level Cross-Validation for Medical Data[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ki-smile/trustcv/blob/main/notebooks/02_Patient_Level_CV.ipynb)## 🎯 Learning ObjectivesAfter completing this notebook, you will be able to:- Understand why patient-level splitting is critical in ML- Identify and prevent patient data leakage- Implement grouped cross-validation strategies- Handle multi-visit and longitudinal patient data- Apply stratified grouped CV for imbalanced medical datasets---

## 1. The Problem: Patient Data LeakageIn medical datasets, we often have:- **Multiple records per patient** (visits, tests, images)- **Correlated measurements** within patients- **Patient-specific patterns** that models can memorizeIf we split randomly, the same patient appears in both training and test sets, leading to:- ⚠️ **Overly optimistic performance estimates**- ⚠️ **Models that fail on new patients**- ⚠️ **Invalid clinical validation**

In [ ]:
# Install trustcv (uncomment for Colab)# !pip install trustcvimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom trustcv import KFoldMedical, GroupKFold, cross_val_scorefrom sklearn.ensemble import RandomForestClassifierfrom sklearn.preprocessing import StandardScalerimport warningswarnings.filterwarnings('ignore')# Set visualization styleplt.style.use('seaborn-v0_8-darkgrid')colors = ['#870052', '#FF876F', '#4F0433', '#EDF4F4']  # KI colorssns.set_palette(colors)print("✅ Libraries loaded successfully!")

## 2. Creating Realistic Patient DataLet's simulate a realistic medical scenario: **Diabetic patients with multiple hospital visits**

In [ ]:
# Generate realistic patient datanp.random.seed(42)def generate_patient_data(n_patients=200, visits_per_patient=(1, 10)):    """    Generate medical data with multiple records per patient    Simulates diabetic patient readmissions    """    data = []        for patient_id in range(n_patients):        # Patient-specific characteristics (constant across visits)        age = np.random.normal(60, 15)        gender = np.random.choice([0, 1])  # 0: Female, 1: Male        baseline_hba1c = np.random.normal(7.5, 2)  # Baseline HbA1c        genetic_risk = np.random.random()  # Hidden genetic factor                # Number of visits for this patient        n_visits = np.random.randint(visits_per_patient[0], visits_per_patient[1])                for visit in range(n_visits):            # Visit-specific measurements (vary across visits)            days_since_baseline = visit * np.random.randint(30, 180)            current_hba1c = baseline_hba1c + np.random.normal(0, 0.5)  # Fluctuates            glucose = 100 + current_hba1c * 15 + np.random.normal(0, 20)            bmi = 25 + age/20 + np.random.normal(0, 3)            medications = min(10, max(1, int(current_hba1c - 5) + np.random.poisson(2)))                        # Emergency visit (outcome) - influenced by patient factors            risk = (                0.1 * (current_hba1c > 8) +                0.2 * (glucose > 180) +                0.15 * (medications > 5) +                0.25 * genetic_risk +  # Patient-specific risk                0.1 * (age > 70)            )            emergency = 1 if np.random.random() < risk else 0                        data.append({                'patient_id': f'P{patient_id:04d}',                'visit_num': visit + 1,                'age': age,                'gender': gender,                'days_since_baseline': days_since_baseline,                'hba1c': current_hba1c,                'glucose': glucose,                'bmi': bmi,                'num_medications': medications,                'emergency_visit': emergency            })        return pd.DataFrame(data)# Generate datasetdf = generate_patient_data(n_patients=200)print(f"📊 Generated dataset with {len(df)} records from {df['patient_id'].nunique()} patients")print(f"\n📈 Records per patient:")print(df['patient_id'].value_counts().describe())# Display sampleprint("\n🔍 Sample of the data:")df.head(10)

In [ ]:
# Visualize patient data distributionfig, axes = plt.subplots(2, 2, figsize=(12, 10))# Records per patientvisit_counts = df['patient_id'].value_counts().valuesaxes[0, 0].hist(visit_counts, bins=15, color=colors[0], edgecolor='black', alpha=0.7)axes[0, 0].set_xlabel('Number of Visits')axes[0, 0].set_ylabel('Number of Patients')axes[0, 0].set_title('Distribution of Visits per Patient')axes[0, 0].axvline(visit_counts.mean(), color=colors[1], linestyle='--',                    label=f'Mean: {visit_counts.mean():.1f}')axes[0, 0].legend()# Outcome distributionoutcome_by_patient = df.groupby('patient_id')['emergency_visit'].mean()axes[0, 1].hist(outcome_by_patient, bins=20, color=colors[1], edgecolor='black', alpha=0.7)axes[0, 1].set_xlabel('Emergency Visit Rate per Patient')axes[0, 1].set_ylabel('Number of Patients')axes[0, 1].set_title('Patient-Level Outcome Distribution')# Feature correlation within patientspatient_sample = df[df['patient_id'].isin(df['patient_id'].unique()[:5])]for pid in patient_sample['patient_id'].unique():    patient_data = patient_sample[patient_sample['patient_id'] == pid]    axes[1, 0].plot(patient_data['visit_num'], patient_data['hba1c'],                     'o-', label=pid, alpha=0.7)axes[1, 0].set_xlabel('Visit Number')axes[1, 0].set_ylabel('HbA1c Level')axes[1, 0].set_title('HbA1c Trajectory for Sample Patients')axes[1, 0].legend()# Overall outcome rateoutcome_dist = df['emergency_visit'].value_counts()axes[1, 1].pie(outcome_dist, labels=['No Emergency', 'Emergency'],                colors=[colors[2], colors[1]], autopct='%1.1f%%')axes[1, 1].set_title('Overall Emergency Visit Rate')plt.tight_layout()plt.show()print(f"\n📊 Key Statistics:")print(f"- Total records: {len(df)}")print(f"- Unique patients: {df['patient_id'].nunique()}")print(f"- Avg visits per patient: {len(df) / df['patient_id'].nunique():.1f}")print(f"- Overall emergency rate: {df['emergency_visit'].mean():.1%}")print(f"- Patients with emergencies: {(outcome_by_patient > 0).mean():.1%}")

## 3. Demonstrating Data Leakage with Random SplitLet's see what happens when we ignore patient grouping:

In [ ]:
# Prepare features and labelsfeature_cols = ['age', 'gender', 'hba1c', 'glucose', 'bmi', 'num_medications']X = df[feature_cols].valuesy = df['emergency_visit'].valuespatient_ids = df['patient_id'].values# Compare random split vs grouped splitfrom sklearn.model_selection import train_test_splitdef check_patient_overlap(train_patients, test_patients):    """Check if same patients appear in train and test"""    train_set = set(train_patients)    test_set = set(test_patients)    overlap = train_set.intersection(test_set)    return len(overlap), len(overlap) / len(train_set.union(test_set))# Method 1: Random split (WRONG!)X_train_random, X_test_random, y_train_random, y_test_random, ids_train_random, ids_test_random = train_test_split(    X, y, patient_ids, test_size=0.2, random_state=42)overlap_count, overlap_pct = check_patient_overlap(ids_train_random, ids_test_random)print("❌ RANDOM SPLIT (Wrong for grouped data):")print(f"   - Patients in training: {len(set(ids_train_random))}")print(f"   - Patients in test: {len(set(ids_test_random))}")print(f"   - Overlapping patients: {overlap_count} ({overlap_pct:.1%})")print(f"   - Result: Same patient in both sets - DATA LEAKAGE!\n")# Method 2: Patient-level split (CORRECT!)unique_patients = df['patient_id'].unique()train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)train_mask = df['patient_id'].isin(train_patients)test_mask = df['patient_id'].isin(test_patients)X_train_grouped = X[train_mask]X_test_grouped = X[test_mask]y_train_grouped = y[train_mask]y_test_grouped = y[test_mask]ids_train_grouped = patient_ids[train_mask]ids_test_grouped = patient_ids[test_mask]overlap_count, overlap_pct = check_patient_overlap(ids_train_grouped, ids_test_grouped)print("✅ PATIENT-LEVEL SPLIT (Correct):")print(f"   - Patients in training: {len(set(ids_train_grouped))}")print(f"   - Patients in test: {len(set(ids_test_grouped))}")print(f"   - Overlapping patients: {overlap_count} ({overlap_pct:.1%})")print(f"   - Result: No patient overlap - NO LEAKAGE!")

In [ ]:
# Compare model performance with both methodsfrom sklearn.metrics import roc_auc_score, accuracy_score# Train modelsrf_random = RandomForestClassifier(n_estimators=100, random_state=42)rf_grouped = RandomForestClassifier(n_estimators=100, random_state=42)# Fit on different splitsrf_random.fit(X_train_random, y_train_random)rf_grouped.fit(X_train_grouped, y_train_grouped)# Evaluatescores_comparison = pd.DataFrame({    'Split Method': ['Random Split (Leakage)', 'Patient-Level Split (No Leakage)'],    'Train Accuracy': [        accuracy_score(y_train_random, rf_random.predict(X_train_random)),        accuracy_score(y_train_grouped, rf_grouped.predict(X_train_grouped))    ],    'Test Accuracy': [        accuracy_score(y_test_random, rf_random.predict(X_test_random)),        accuracy_score(y_test_grouped, rf_grouped.predict(X_test_grouped))    ],    'Test AUC': [        roc_auc_score(y_test_random, rf_random.predict_proba(X_test_random)[:, 1]),        roc_auc_score(y_test_grouped, rf_grouped.predict_proba(X_test_grouped)[:, 1])    ]})print("\n📊 Performance Comparison:")print(scores_comparison.to_string(index=False))# Visualize the differencefig, ax = plt.subplots(figsize=(10, 6))x = np.arange(len(scores_comparison))width = 0.25ax.bar(x - width, scores_comparison['Train Accuracy'], width, label='Train Accuracy', color=colors[0])ax.bar(x, scores_comparison['Test Accuracy'], width, label='Test Accuracy', color=colors[1])ax.bar(x + width, scores_comparison['Test AUC'], width, label='Test AUC', color=colors[2])ax.set_xlabel('Split Method')ax.set_ylabel('Score')ax.set_title('Impact of Data Leakage on Model Performance')ax.set_xticks(x)ax.set_xticklabels(scores_comparison['Split Method'])ax.legend()ax.grid(True, alpha=0.3)# Add warning annotationax.annotate('⚠️ Overoptimistic!', xy=(0, scores_comparison.iloc[0]['Test Accuracy']),             xytext=(0, scores_comparison.iloc[0]['Test Accuracy'] + 0.05),            arrowprops=dict(arrowstyle='->', color='red'),            fontsize=12, color='red', ha='center')plt.tight_layout()plt.show()print("\n⚠️ WARNING: Random split shows inflated performance due to data leakage!")print(f"Performance inflation: {(scores_comparison.iloc[0]['Test Accuracy'] - scores_comparison.iloc[1]['Test Accuracy']) / scores_comparison.iloc[1]['Test Accuracy'] * 100:.1f}%")

## 4. Implementing Group K-Fold Cross-ValidationNow let's use proper grouped cross-validation:

In [ ]:
from trustcv import GroupKFoldMedical, StratifiedGroupKFold# Set up different CV strategiescv_strategies = {    'Regular K-Fold (Wrong)': KFoldMedical(n_splits=5, shuffle=True, random_state=42),    'Group K-Fold (Correct)': GroupKFoldMedical(n_splits=5)}# Compare CV strategiescv_results = {}for cv_name, cv in cv_strategies.items():    print(f"\n🔄 Testing: {cv_name}")        fold_scores = []    patient_leakage = []        # Perform cross-validation    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=patient_ids)):        # Check for patient leakage        train_patients = patient_ids[train_idx]        test_patients = patient_ids[test_idx]        overlap, overlap_pct = check_patient_overlap(train_patients, test_patients)        patient_leakage.append(overlap_pct)                # Train and evaluate        X_train_cv, X_test_cv = X[train_idx], X[test_idx]        y_train_cv, y_test_cv = y[train_idx], y[test_idx]                rf = RandomForestClassifier(n_estimators=50, random_state=42)        rf.fit(X_train_cv, y_train_cv)        score = accuracy_score(y_test_cv, rf.predict(X_test_cv))        fold_scores.append(score)                print(f"   Fold {fold_idx + 1}: Accuracy={score:.3f}, Patient Overlap={overlap_pct:.1%}")        cv_results[cv_name] = {        'scores': fold_scores,        'mean': np.mean(fold_scores),        'std': np.std(fold_scores),        'leakage': np.mean(patient_leakage)    }        print(f"   Mean Accuracy: {np.mean(fold_scores):.3f} ± {np.std(fold_scores):.3f}")    print(f"   Average Patient Leakage: {np.mean(patient_leakage):.1%}")

In [ ]:
# Visualize CV comparisonfig, axes = plt.subplots(1, 2, figsize=(12, 5))# Box plot of scorescv_scores_df = pd.DataFrame({    name: results['scores']     for name, results in cv_results.items()})cv_scores_df.boxplot(ax=axes[0], color={'boxes': colors[0], 'whiskers': colors[1]})axes[0].set_ylabel('Accuracy')axes[0].set_title('Cross-Validation Score Distribution')axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')# Leakage comparisonleakage_data = [results['leakage'] * 100 for results in cv_results.values()]bars = axes[1].bar(range(len(cv_results)), leakage_data, color=[colors[2], colors[0]])axes[1].set_ylabel('Patient Leakage (%)')axes[1].set_title('Data Leakage by CV Method')axes[1].set_xticks(range(len(cv_results)))axes[1].set_xticklabels(cv_results.keys(), rotation=45, ha='right')# Add value labelsfor bar, val in zip(bars, leakage_data):    if val > 0:        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,                    f'{val:.1f}%', ha='center', color='red', fontweight='bold')    else:        axes[1].text(bar.get_x() + bar.get_width()/2, 1,                    '✓ No Leakage', ha='center', color='green', fontweight='bold')plt.tight_layout()plt.show()print("\n📊 Summary:")for name, results in cv_results.items():    print(f"{name}:")    print(f"  - Mean Score: {results['mean']:.3f} ± {results['std']:.3f}")    print(f"  - Patient Leakage: {results['leakage']:.1%}")    print(f"  - Valid for Clinical Use: {'✅ Yes' if results['leakage'] == 0 else '❌ No'}")

## 5. Using trustcv for Patient-Level ValidationNow let's use the trustcv package for automated patient-aware validation:

In [ ]:
# Import trustcv componentsfrom trustcv import MedicalValidatorfrom trustcv.splitters import PatientGroupKFold, StratifiedGroupKFoldfrom trustcv.checkers import DataLeakageChecker# Initialize medical validatorvalidator = MedicalValidator(    method='patient_grouped_kfold',    n_splits=5,    check_leakage=True,    check_balance=True)# Create pipelinefrom sklearn.pipeline import Pipelinepipeline = Pipeline([    ('scaler', StandardScaler()),    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])print("🏥 Running medical-aware cross-validation...")print("This ensures:")print("  ✓ No patient appears in multiple folds")print("  ✓ Automatic leakage detection")print("  ✓ Clinical metrics calculation")print("  ✓ Confidence intervals\n")# Perform validationresults = validator.fit_validate(    model=pipeline,    X=pd.DataFrame(X, columns=feature_cols),    y=pd.Series(y),    patient_ids=pd.Series(patient_ids))# Display resultsprint(results.summary())

## 6. Stratified Group K-Fold for Imbalanced DataWhen you have both:- Multiple records per patient (need grouping)- Class imbalance (need stratification)Use **Stratified Group K-Fold**:

In [ ]:
# Create imbalanced subset# Select patients with different outcome ratespatient_outcomes = df.groupby('patient_id')['emergency_visit'].mean()rare_patients = patient_outcomes[patient_outcomes > 0.5].index[:10]  # High riskcommon_patients = patient_outcomes[patient_outcomes == 0].index[:90]  # No eventsselected_patients = list(rare_patients) + list(common_patients)imbalanced_df = df[df['patient_id'].isin(selected_patients)]print(f"📊 Imbalanced Dataset:")print(f"  - Total records: {len(imbalanced_df)}")print(f"  - Unique patients: {imbalanced_df['patient_id'].nunique()}")print(f"  - Class distribution:")print(imbalanced_df['emergency_visit'].value_counts(normalize=True))# Prepare dataX_imb = imbalanced_df[feature_cols].valuesy_imb = imbalanced_df['emergency_visit'].valuesgroups_imb = imbalanced_df['patient_id'].values

In [ ]:
# Compare regular GroupKFold vs StratifiedGroupKFoldfrom trustcv.splitters import StratifiedGroupKFold# Regular GroupKFoldgkf = GroupKFoldMedical(n_splits=5)# Stratified GroupKFoldsgkf = StratifiedGroupKFold(n_splits=5, random_state=42)# Compare class distributionsfig, axes = plt.subplots(2, 5, figsize=(15, 6))# Regular GroupKFoldfor fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_imb, y_imb, groups_imb)):    test_dist = y_imb[test_idx]    positive_rate = test_dist.mean()        axes[0, fold_idx].bar([0, 1], [1-positive_rate, positive_rate],                           color=[colors[0], colors[1]])    axes[0, fold_idx].set_ylim([0, 1])    axes[0, fold_idx].set_title(f'Fold {fold_idx+1}\n{positive_rate:.1%} positive')    axes[0, fold_idx].set_xticks([0, 1])    axes[0, fold_idx].set_xticklabels(['Neg', 'Pos'])        if fold_idx == 0:        axes[0, fold_idx].set_ylabel('Regular\nGroupKFold', fontweight='bold')# Stratified GroupKFoldfor fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X_imb, y_imb, groups_imb)):    test_dist = y_imb[test_idx]    positive_rate = test_dist.mean()        axes[1, fold_idx].bar([0, 1], [1-positive_rate, positive_rate],                           color=[colors[0], colors[1]])    axes[1, fold_idx].set_ylim([0, 1])    axes[1, fold_idx].set_title(f'Fold {fold_idx+1}\n{positive_rate:.1%} positive')    axes[1, fold_idx].set_xticks([0, 1])    axes[1, fold_idx].set_xticklabels(['Neg', 'Pos'])        if fold_idx == 0:        axes[1, fold_idx].set_ylabel('Stratified\nGroupKFold', fontweight='bold')plt.suptitle('Class Distribution Across Folds: Regular vs Stratified Group K-Fold',              fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n📊 Analysis:")print("- Regular GroupKFold: Class distribution varies significantly across folds")print("- Stratified GroupKFold: Maintains consistent class distribution")print("- Both methods: Ensure no patient appears in multiple folds")

## 7. Advanced: Nested Patient Data (Hospital → Department → Patient)Sometimes we have hierarchical grouping:

In [ ]:
# Add hospital and department informationnp.random.seed(42)# Simulate multi-site datahospitals = ['Hospital A', 'Hospital B', 'Hospital C']departments = ['Cardiology', 'Endocrinology', 'Internal Medicine']# Assign patients to hospitals and departmentspatient_hospital_map = {}patient_dept_map = {}for patient in df['patient_id'].unique():    patient_hospital_map[patient] = np.random.choice(hospitals)    patient_dept_map[patient] = np.random.choice(departments)df['hospital'] = df['patient_id'].map(patient_hospital_map)df['department'] = df['patient_id'].map(patient_dept_map)print("🏥 Multi-Site Data Structure:")print("\nPatients per Hospital:")print(df.groupby('hospital')['patient_id'].nunique())print("\nPatients per Department:")print(df.groupby('department')['patient_id'].nunique())# Visualize hierarchical structurehierarchy_summary = df.groupby(['hospital', 'department'])['patient_id'].nunique().reset_index()hierarchy_summary.columns = ['Hospital', 'Department', 'Patients']fig, ax = plt.subplots(figsize=(10, 6))pivot_data = hierarchy_summary.pivot(index='Hospital', columns='Department', values='Patients')pivot_data.plot(kind='bar', stacked=True, ax=ax, color=colors[:3])ax.set_ylabel('Number of Patients')ax.set_title('Patient Distribution Across Hospitals and Departments')ax.legend(title='Department')plt.xticks(rotation=0)plt.tight_layout()plt.show()

In [ ]:
# Different splitting strategies for hierarchical dataprint("🔄 Hierarchical Splitting Strategies:\n")# Strategy 1: Split by Hospital (entire hospitals in train or test)hospital_train = ['Hospital A', 'Hospital B']hospital_test = ['Hospital C']train_mask = df['hospital'].isin(hospital_train)test_mask = df['hospital'].isin(hospital_test)print("1️⃣ Hospital-Level Split:")print(f"   Training: {hospital_train} ({train_mask.sum()} records)")print(f"   Testing: {hospital_test} ({test_mask.sum()} records)")print(f"   Use case: Test model generalization to new hospitals\n")# Strategy 2: Split by Department within hospitalsdept_train = ['Cardiology', 'Endocrinology']dept_test = ['Internal Medicine']train_mask = df['department'].isin(dept_train)test_mask = df['department'].isin(dept_test)print("2️⃣ Department-Level Split:")print(f"   Training: {dept_train} ({train_mask.sum()} records)")print(f"   Testing: {dept_test} ({test_mask.sum()} records)")print(f"   Use case: Test model across different specialties\n")# Strategy 3: Patient-level within each hospitalprint("3️⃣ Patient-Level Split (within hospitals):")for hospital in hospitals:    hospital_data = df[df['hospital'] == hospital]    n_patients = hospital_data['patient_id'].nunique()    print(f"   {hospital}: {n_patients} patients → {int(n_patients*0.8)} train, {int(n_patients*0.2)} test")print(f"   Use case: Standard validation within each site")

## 8. Best Practices Summary### ✅ DO's:1. **Always use GroupKFold** when you have multiple records per patient2. **Check for patient overlap** between train and test sets3. **Use StratifiedGroupKFold** for imbalanced grouped data4. **Consider hierarchical structure** (hospital → department → patient)5. **Document your splitting strategy** for reproducibility### ❌ DON'Ts:1. **Never use random split** with grouped medical data2. **Don't ignore patient correlations** within the data3. **Don't mix records** from the same patient across train/test4. **Don't assume independence** when patients have multiple visits### 🔍 Always Check:- Number of unique patients vs total records- Distribution of records per patient- Patient overlap between folds- Class distribution preservation- Site/department effects

In [ ]:
# Final validation pipeline with all best practicesprint("🎯 Complete Patient-Level Validation Pipeline\n")# 1. Data inspectionprint("Step 1: Data Inspection")print(f"  - Records: {len(df)}")print(f"  - Patients: {df['patient_id'].nunique()}")print(f"  - Records/Patient: {len(df)/df['patient_id'].nunique():.1f}")print(f"  - Outcome rate: {df['emergency_visit'].mean():.1%}\n")# 2. Choose appropriate CV strategyprint("Step 2: Select CV Strategy")if len(df) > df['patient_id'].nunique():    print("  ✓ Multiple records per patient detected")    if df['emergency_visit'].mean() < 0.3:        print("  ✓ Class imbalance detected")        print("  → Using: StratifiedGroupKFold\n")        cv_strategy = 'stratified_group_kfold'    else:        print("  → Using: GroupKFold\n")        cv_strategy = 'group_kfold'else:    print("  → Using: StratifiedKFold\n")    cv_strategy = 'stratified_kfold'# 3. Initialize validatorprint("Step 3: Initialize Medical Validator")validator = MedicalValidator(    method=cv_strategy,    n_splits=5,    check_leakage=True,    check_balance=True)print(f"  ✓ Validator configured with {cv_strategy}\n")# 4. Create pipelineprint("Step 4: Create ML Pipeline")pipeline = Pipeline([    ('scaler', StandardScaler()),    ('classifier', RandomForestClassifier(        n_estimators=100,        max_depth=5,  # Prevent overfitting        min_samples_leaf=20,  # Ensure generalization        random_state=42,        class_weight='balanced'  # Handle imbalance    ))])print("  ✓ Pipeline created with preprocessing and classifier\n")# 5. Run validationprint("Step 5: Run Cross-Validation")results = validator.fit_validate(    model=pipeline,    X=pd.DataFrame(X, columns=feature_cols),    y=pd.Series(y),    patient_ids=pd.Series(patient_ids))# 6. Display resultsprint("\n" + "="*60)print("FINAL RESULTS")print("="*60)print(results.summary())print("\n✅ Validation Complete!")print("This model can be trusted for clinical validation.")

## 9. Exercise: Implement Your Own Patient-Level CVTry these exercises to reinforce your learning:

In [ ]:
# Exercise 1: Create a custom patient splitterdef create_patient_time_split(df, patient_col, time_col, test_size=0.2):    """    TODO: Create train/test split where:    - No patient appears in both sets    - Test set contains most recent patients    """    # Your code here    pass# Exercise 2: Implement leave-one-patient-out CVdef leave_one_patient_out_cv(X, y, patient_ids):    """    TODO: Implement LOPO CV where each patient is test set once    """    # Your code here    pass# Exercise 3: Calculate patient-level metricsdef patient_level_metrics(y_true, y_pred, patient_ids):    """    TODO: Calculate accuracy per patient (not per record)    """    # Your code here    passprint("📝 Complete the exercises above to practice patient-level CV!")

## 🎓 Key Takeaways1. **Patient data leakage is a critical issue** that leads to overoptimistic results2. **Always use GroupKFold** when you have multiple records per patient3. **StratifiedGroupKFold** handles both grouping and class imbalance4. **Check for overlap** between train and test sets5. **Consider hierarchical structure** in multi-site studies6. **trustcv automates** these best practices for you---### 🚀 Next StepsContinue with:- **[03_Temporal_Medical.ipynb](03_Temporal_Medical.ipynb)** - Time series validation for medical data- **[04_Nested_CV.ipynb](04_Nested_CV.ipynb)** - Hyperparameter tuning without leakage---*Created with trustcv - Trustworthy Cross-Validation Toolkit*